# 31-QLib 框架入门

> 模块 4.1 ML量化 | 理解量化平台架构，手写 QLib 核心流水线

## 学习目标

- 理解 QLib 的四大核心模块：Data / Model / Strategy / Executor
- 能用手写代码复现 QLib 的模块间协作流程
- 对比传统因子（线性回归）与 ML 因子（LightGBM）在方向预测上的差异
- 搭建一个可运行的 mini-QLib 流水线，跑通数据→因子→模型→回测

## 环境依赖

本 notebook 使用仓库已有依赖：`numpy`、`pandas`、`matplotlib`。

QLib 官方框架（`pyqlib`）需要 `torch` 等额外依赖，安装成本较高。本课的核心目标是**理解架构设计**，所以用纯 numpy/pandas 手写 QLib 四大组件，外部数据请求同样使用 `try/except + 模拟数据` 降级。

如果本地已安装 QLib（`pip install pyqlib`），代码会优先使用真实 QLib；否则自动降级到手写组件，学习目标不受影响。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# 尝试导入 QLib
QLIB_AVAILABLE = False
try:
    import qlib
    from qlib.data import D
    QLIB_AVAILABLE = True
    print("✅ QLib 已安装，将使用真实 QLib API")
except ImportError:
    print("⚠️ QLib 未安装，使用手写 mini-QLib 组件（学习架构思想不受影响）")
    print("  如需安装: pip install pyqlib")

## 1. QLib 架构全景

QLib 是微软发布的 AI 量化平台，核心设计思想是**模块化 + 可扩展**。

### 四大核心模块

```text
┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐
│  Data    │ → │  Model   │ → │ Strategy │ → │ Executor │
│ 数据准备  │   │ 模型训练  │   │ 策略信号  │   │ 回测评估  │
└──────────┘   └──────────┘   └──────────┘   └──────────┘
     ↓               ↓              ↓              ↓
  因子构造      学习 f(X)→y     信号生成       绩效计算
  标签对齐      样本外预测       仓位管理       归因分析
```

| 模块 | 职责 | 输入 | 输出 |
|------|------|------|------|
| **Data** | 数据加载、因子构造、标签计算、时间切分 | 原始行情 | 结构化数据集 (X, y, 时间索引) |
| **Model** | 模型训练与预测，支持线性/树模型/深度学习 | 特征矩阵 X, 标签 y | 预测值 pred (收益或方向概率) |
| **Strategy** | 将模型预测转化为交易信号和仓位 | 预测值, 约束条件 | 交易信号 (buy/sell/hold) |
| **Executor** | 执行回测，计算绩效指标 | 信号, 价格序列 | 净值曲线、Sharpe、回撤等 |

### 为什么需要这种设计？

1. **解耦**：换模型不改数据，换策略不改模型
2. **可复现**：每一步的输出都有明确接口，方便调试
3. **可扩展**：新增一个模型只需实现 Model 接口，其他模块不变

接下来我们逐个实现这四个模块。

## 2. 数据准备：模拟价格序列

为了让 notebook 在任何环境都能跑通，先构造一段模拟的价格序列。真实使用时替换为 AKShare/本地数据库数据即可。

In [ ]:
def generate_price_data(n=500):
    """生成模拟价格序列：趋势 + 波动聚集 + 噪声"""
    dates = pd.date_range("2020-01-01", periods=n, freq="B")
    # 对数收益率
    np.random.seed(RANDOM_SEED)
    mu = 0.0003  # 日均收益约 0.03%
    # 波动聚集：前半段低波动，后半段高波动
    sigma = np.where(np.arange(n) < n // 2, 0.01, 0.02)
    noise = np.random.randn(n) * sigma
    returns = mu + noise
    price = 100 * np.exp(np.cumsum(returns))

    df = pd.DataFrame({
        "date": dates,
        "close": price,
        "return": np.append([0], returns[1:]),
    })
    df.set_index("date", inplace=True)
    return df

price_df = generate_price_data(500)
print(f"数据范围: {price_df.index[0].date()} ~ {price_df.index[-1].date()}")
print(f"交易日数: {len(price_df)}")

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(price_df.index, price_df["close"], linewidth=1)
axes[0].set_title("模拟价格序列")
axes[0].set_ylabel("价格")
axes[1].plot(price_df.index, price_df["return"], linewidth=0.5, alpha=0.7)
axes[1].set_title("日收益率")
axes[1].set_ylabel("收益率")
plt.tight_layout()
plt.show()

## 3. Data 模块：因子构造 + 标签 + 时间切分

Data 模块是 QLib 流水线的入口。它负责三件事：

1. **因子构造**：从价格/成交量衍生特征
2. **标签计算**：构造未来收益标签（注意时序对齐，避免前视偏差）
3. **时间切分**：按时间顺序划分训练集和测试集

这是最容易引入前视偏差的环节，必须小心处理。

In [ ]:
class QLibData:
    """mini-QLib Data 模块：因子构造 + 标签 + 时间切分"""

    def __init__(self, price_df, horizon=5):
        self.price_df = price_df.copy()
        self.horizon = horizon
        self.feature_names = []
        self.label_name = f"future_ret_{horizon}d"

    def build_factors(self):
        """构造技术面因子"""
        df = self.price_df.copy()

        # 动量因子
        df["ret_5d"] = df["close"].pct_change(5)
        df["ret_20d"] = df["close"].pct_change(20)

        # 波动率因子
        df["vol_5d"] = df["return"].rolling(5).std()
        df["vol_20d"] = df["return"].rolling(20).std()

        # 均线偏离因子
        ma_20 = df["close"].rolling(20).mean()
        df["ma_dev_20d"] = (df["close"] - ma_20) / ma_20

        # 成交量代理（用收益率绝对值代替，模拟数据没有成交量）
        df["turnover_proxy"] = np.abs(df["return"]) / df["vol_20d"].replace(0, np.nan)

        self.feature_names = ["ret_5d", "ret_20d", "vol_5d", "vol_20d", "ma_dev_20d", "turnover_proxy"]
        return df

    def build_labels(self, df):
        """构造未来收益标签（方向）

        注意：用 shift(-horizon) 没问题——标签就是未来值，模型用当前特征预测它。
        """
        df[self.label_name] = df["close"].shift(-self.horizon) / df["close"] - 1
        df["target"] = (df[self.label_name] > 0).astype(int)
        return df

    def prepare(self, train_ratio=0.7):
        """完整数据准备流水线：因子 → 标签 → 清洗 → 切分"""
        df = self.build_factors()
        df = self.build_labels(df)
        df = df.dropna()

        split = int(len(df) * train_ratio)
        train = df.iloc[:split]
        test = df.iloc[split:]

        return {
            "full": df,
            "X_train": train[self.feature_names].values,
            "y_train": train["target"].values,
            "X_test": test[self.feature_names].values,
            "y_test": test["target"].values,
            "test_df": test,
            "train_df": train,
            "feature_names": self.feature_names,
        }

# 运行 Data 模块
data_module = QLibData(price_df, horizon=5)
dataset = data_module.prepare(train_ratio=0.7)

print(f"特征数量: {len(dataset['feature_names'])}")
print(f"特征名称: {dataset['feature_names']}")
print(f"训练集: {len(dataset['X_train'])} 条")
print(f"测试集: {len(dataset['X_test'])} 条")
print(f"训练集标签分布: 上涨={dataset['y_train'].sum()}, 下跌={len(dataset['y_train']) - dataset['y_train'].sum()}")

## 4. Model 模块：从线性回归到 LightGBM

Model 模块负责学习映射 $f: X \to y$。QLib 支持多种模型，这里我们实现两种：
- **线性回归（传统因子方法）**：用 OLS 估计因子权重
- **梯度提升树（ML 因子方法）**：如果 LightGBM 可用就用，不可用就用 sklearn 决策树作为替代

In [ ]:
class QLibModel:
    """mini-QLib Model 模块：训练 + 预测"""

    def __init__(self, model_type="linear"):
        self.model_type = model_type
        self.model = None
        self.feature_importance = None

    def fit(self, X, y, feature_names=None):
        """训练模型"""
        X = np.array(X)
        y = np.array(y)

        if self.model_type == "linear":
            # OLS: beta = (X^T X)^{-1} X^T y
            X_aug = np.column_stack([np.ones(len(X)), X])
            beta = np.linalg.lstsq(X_aug, y, rcond=None)[0]
            self.model = beta
            self.feature_importance = np.abs(beta[1:])  # 跳过截距

        elif self.model_type == "tree":
            try:
                import lightgbm as lgb
                self.model = lgb.LGBMClassifier(
                    n_estimators=100, max_depth=5,
                    random_state=RANDOM_SEED, verbose=-1
                )
                self.model.fit(X, y)
                self.feature_importance = self.model.feature_importances_
                print("✅ 使用 LightGBM")
            except ImportError:
                from sklearn.tree import DecisionTreeClassifier
                self.model = DecisionTreeClassifier(
                    max_depth=5, random_state=RANDOM_SEED
                )
                self.model.fit(X, y)
                self.feature_importance = self.model.feature_importances_
                print("⚠️ LightGBM 不可用，降级为 sklearn 决策树")

    def predict(self, X):
        """预测"""
        X = np.array(X)
        if self.model_type == "linear":
            X_aug = np.column_stack([np.ones(len(X)), X])
            return X_aug @ self.model
        else:
            return self.model.predict_proba(X)[:, 1]

# 训练两个模型
linear_model = QLibModel("linear")
linear_model.fit(dataset["X_train"], dataset["y_train"], dataset["feature_names"])
pred_linear = linear_model.predict(dataset["X_test"])

tree_model = QLibModel("tree")
tree_model.fit(dataset["X_train"], dataset["y_train"], dataset["feature_names"])
pred_tree = tree_model.predict(dataset["X_test"])

# 特征重要性对比
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = dataset["feature_names"]

axes[0].barh(names, linear_model.feature_importance)
axes[0].set_title("线性模型 |β| 系数")
axes[0].invert_yaxis()

axes[1].barh(names, tree_model.feature_importance)
axes[1].set_title("树模型 特征重要性")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 5. Strategy 模块：从预测到信号

模型输出的是一个连续值（或概率），Strategy 负责将其转化为可交易的信号。

一个经典的量化策略逻辑：
- 预测值 > 阈值 → 做多
- 预测值 < -阈值 → 做空（或空仓）
- 中间地带 → 空仓

阈值的选择需要平衡信号数量和信号质量。

In [ ]:
class QLibStrategy:
    """mini-QLib Strategy 模块：预测值 → 交易信号"""

    def __init__(self, top_quantile=0.3):
        self.top_quantile = top_quantile

    def generate_signals(self, predictions, index=None):
        """将预测值转化为信号

        - Top quantile: 做多 (signal = 1)
        - Bottom quantile: 做空/空仓 (signal = -1)
        - 中间: 空仓 (signal = 0)
        """
        preds = np.array(predictions)
        top_thresh = np.quantile(preds, 1 - self.top_quantile)
        bottom_thresh = np.quantile(preds, self.top_quantile)

        signals = np.zeros_like(preds)
        signals[preds >= top_thresh] = 1
        signals[preds <= bottom_thresh] = -1

        if index is not None:
            return pd.Series(signals, index=index)
        return pd.Series(signals)

strategy = QLibStrategy(top_quantile=0.3)
signals_linear = strategy.generate_signals(pred_linear, dataset["test_df"].index)
signals_tree = strategy.generate_signals(pred_tree, dataset["test_df"].index)

print(f"线性模型信号分布:\n{signals_linear.value_counts().to_string()}")
print(f"\n树模型信号分布:\n{signals_tree.value_counts().to_string()}")

## 6. Executor 模块：回测与绩效评估

Executor 是流水线的最后一环，它接收交易信号和价格序列，模拟交易过程并计算绩效指标。

QLib 的 Executor 支持复杂撮合逻辑（限价单/市价单、手续费、滑点等），这里实现简化版以聚焦于核心思想。

In [ ]:
class QLibExecutor:
    """mini-QLib Executor 模块：回测 + 绩效评估"""

    def __init__(self, returns, signals, fee_rate=0.001):
        self.returns = np.array(returns)
        self.signals = np.array(signals)
        self.fee_rate = fee_rate

    def backtest(self):
        """执行回测：策略收益 = 信号 × 市场收益 - 手续费"""
        n = len(self.returns)
        # 手续费：信号变化时扣费
        signal_diff = np.abs(np.diff(self.signals, prepend=0))
        strategy_returns = self.signals * self.returns - signal_diff * self.fee_rate
        strategy_nav = np.cumprod(1 + strategy_returns)
        return strategy_returns, strategy_nav

    def evaluate(self):
        """计算核心绩效指标"""
        strategy_returns, strategy_nav = self.backtest()

        # 年化收益率（假设252个交易日）
        total_return = strategy_nav[-1] - 1
        annual_return = (1 + total_return) ** (252 / len(strategy_returns)) - 1

        # 年化波动率
        annual_vol = np.std(strategy_returns) * np.sqrt(252)

        # Sharpe（假设无风险利率=0）
        sharpe = annual_return / annual_vol if annual_vol > 0 else 0

        # 最大回撤
        peak = np.maximum.accumulate(strategy_nav)
        drawdown = (strategy_nav - peak) / peak
        max_dd = drawdown.min()

        # 胜率
        win_rate = (strategy_returns > 0).mean()

        # 盈亏比
        gains = strategy_returns[strategy_returns > 0]
        losses = strategy_returns[strategy_returns < 0]
        profit_loss_ratio = gains.mean() / abs(losses.mean()) if len(losses) > 0 else np.inf

        return {
            "累计收益": f"{total_return:.2%}",
            "年化收益": f"{annual_return:.2%}",
            "年化波动": f"{annual_vol:.2%}",
            "Sharpe": f"{sharpe:.3f}",
            "最大回撤": f"{max_dd:.2%}",
            "胜率": f"{win_rate:.2%}",
            "盈亏比": f"{profit_loss_ratio:.2f}",
            "nav": strategy_nav,
            "returns": strategy_returns,
        }

# 运行回测
test_returns = dataset["test_df"]["return"].values

executor_linear = QLibExecutor(test_returns, signals_linear.values)
result_linear = executor_linear.evaluate()

executor_tree = QLibExecutor(test_returns, signals_tree.values)
result_tree = executor_tree.evaluate()

# 基准（买入持有）
benchmark_nav = np.cumprod(1 + test_returns)

print("=" * 50)
print("传统因子（线性回归） vs ML 因子（树模型）")
print("=" * 50)
for key in ["累计收益", "年化收益", "年化波动", "Sharpe", "最大回撤", "胜率"]:
    print(f"{key:8s} | 线性: {result_linear[key]:>10s} | 树模型: {result_tree[key]:>10s}")

# 可视化
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(result_linear["nav"], label="线性模型策略", linewidth=1.5)
axes[0].plot(result_tree["nav"], label="树模型策略", linewidth=1.5)
axes[0].plot(benchmark_nav, label="买入持有基准", linewidth=1, linestyle="--", alpha=0.6)
axes[0].set_title("净值曲线对比")
axes[0].legend()
axes[0].set_ylabel("净值")

# 回撤曲线
peak_l = np.maximum.accumulate(result_linear["nav"])
dd_l = (result_linear["nav"] - peak_l) / peak_l
peak_t = np.maximum.accumulate(result_tree["nav"])
dd_t = (result_tree["nav"] - peak_t) / peak_t

axes[1].fill_between(range(len(dd_l)), dd_l * 100, 0, alpha=0.3, label="线性模型")
axes[1].fill_between(range(len(dd_t)), dd_t * 100, 0, alpha=0.3, label="树模型")
axes[1].set_title("回撤曲线 (%)")
axes[1].set_ylabel("回撤")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. 完整流水线串联

上面我们已经分别实现了四个模块。QLib 的核心价值在于把它们**串联成一个可配置的流水线**：

```text
Data.prepare() → Model.fit() → Model.predict() → Strategy.generate_signals() → Executor.evaluate()
```

真实 QLib 中，每一步都是通过 yaml 配置文件来定义的，不同模块之间通过约定好的接口传递数据。

下面我们把这个流程封装成一个函数，方便对比不同配置（换模型、换因子、换参数）的效果。

In [ ]:
def run_full_pipeline(price_df, model_type="linear", horizon=5, top_quantile=0.3):
    """QLib 完整流水线：Data → Model → Strategy → Executor"""
    # Data
    data = QLibData(price_df, horizon=horizon)
    ds = data.prepare()

    # Model
    model = QLibModel(model_type)
    model.fit(ds["X_train"], ds["y_train"], ds["feature_names"])
    preds = model.predict(ds["X_test"])

    # Strategy
    strategy = QLibStrategy(top_quantile=top_quantile)
    signals = strategy.generate_signals(preds, ds["test_df"].index)

    # Executor
    executor = QLibExecutor(ds["test_df"]["return"].values, signals.values)
    result = executor.evaluate()

    return result

# 对比不同配置
configs = [
    ("线性", "linear", 5),
    ("树模型", "tree", 5),
    ("线性-长周期", "linear", 20),
    ("树模型-长周期", "tree", 20),
]

print(f"{'配置':<16s} {'Sharpe':>8s} {'年化收益':>10s} {'最大回撤':>10s}")
print("-" * 50)
for name, mtype, h in configs:
    r = run_full_pipeline(price_df, model_type=mtype, horizon=h)
    print(f"{name:<16s} {r['Sharpe']:>8s} {r['年化收益']:>10s} {r['最大回撤']:>10s}")

## 8. 传统因子 vs ML 因子：关键差异

从上面的实验可以观察到 ML 方法和传统方法在量化中的本质差异：

| 维度 | 传统因子（线性） | ML 因子（树模型） |
|------|-----------------|-------------------|
| **假设** | 线性关系、同方差、因子独立 | 不依赖线性假设，自动捕捉非线性 |
| **因子交互** | 需要手动构造交互项 | 自动学习因子间的交互（如 "高动量+低波动"） |
| **可解释性** | 高（每个 β 有明确含义） | 中低（特征重要性可用，但不如系数直观） |
| **过拟合风险** | 低（参数少，结构简单） | 高（需要严格控制 max_depth / n_estimators） |
| **样本需求** | 小（几百条就够了） | 大（通常需要几千条以上） |
| **调参难度** | 低（基本不需要） | 高（超参数组合多） |
| **典型场景** | Fama-MacBeth 因子检验 | 高频预测、多空组合、另类数据 |

### QLib 的定位

QLib 不是要替代传统因子研究，而是提供一个**框架**让你能公平对比两者：
- 统一的数据接口保证同样的 $X$ 和 $y$ 喂给不同模型
- 统一的回测引擎保证绩效指标可比
- 模块化设计让你可以只换模型，其他部分不变

这就是 "ML 量化的优势"：不是 ML 一定更好，而是你可以用工程化的方式去验证它。

## 9. 小结

这节课我们：

1. **理解了 QLib 的四大模块**：Data(数据) → Model(模型) → Strategy(策略) → Executor(回测)
2. **手写了一个 mini-QLib**：用 numpy/pandas 复现了每个模块的核心逻辑
3. **跑了完整流水线**：从因子构造到绩效评估，对比了线性和树模型
4. **认识了 ML 量化的关键问题**：非线性捕捉 vs 过拟合风险、可解释性 vs 预测力

### 下一步

如果你安装了 QLib，可以尝试：
- 用 `qlib.init()` 初始化真实 QLib 环境
- 把本课的 `QLibData` 替换为 `qlib.data.dataset.DatasetH`
- 导入 QLib 内置的 LightGBM / GRU / Transformer 模型
- 用 QLib 的 `backtest` 模块替换手写 `QLibExecutor`

本课的 mini-QLib 虽然简单，但架构和真实 QLib 完全一致，理解了这个就理解了 QLib 的设计思想。

## 验收清单

- [ ] 能画出 QLib 四大模块的流程图并解释各自的职责
- [ ] 能用手写代码实现 Data → Model → Strategy → Executor 全流程
- [ ] 能比较线性回归和树模型在同一组因子上的表现差异
- [ ] 理解时间顺序切分的重要性（为什么不随机切分）
- [ ] 能说出 ML 量化相对于传统因子投资的 3 个优势和 3 个风险
- [ ] 知道 QLib 中哪个模块最容易引入前视偏差（Data 模块）